For best results, run this notebook with 2x T4 GPUs

In [ ]:
import os
import base64
import subprocess
from kaggle_secrets import UserSecretsClient


In [ ]:
# set this to match your Kaggle notebook's URL slug (also used as the Drive folder name)
NOTEBOOK_NAME = "your-kaggle-notebook"
assert NOTEBOOK_NAME != "your-kaggle-notebook", \
    "Set NOTEBOOK_NAME to your real notebook slug — the placeholder pollutes /kaggle/working and Drive."


In [ ]:
repo = f"/kaggle/working/{NOTEBOOK_NAME}"

if not os.path.exists(repo):
    subprocess.run(["git", "clone", "https://github.com/noshou/APS360.git", repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "pull"], check=True)


In [ ]:
# only needs to be run once per session
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py pyyaml torch-tb-profiler
!apt-get install -y -q git-lfs
!git lfs install
!curl -fsSL https://rclone.org/install.sh | sudo bash

In [ ]:
# ── rclone / Google Drive setup ── run once per session ─────────────────────
# Mirrors checkpoints off-box so a session timeout doesn't lose them.
# Prereq (one-time): configure rclone locally, then paste the contents of
# base64 -w0 ~/.config/rclone/rclone.conf into a Kaggle Secret named RCLONE_CONF
# (Add-ons -> Secrets). REMOTE_NAME is derived automatically from your rclone config.
# we use base64 so we can copy/paste in single line even tho it's multiline token :)

conf_path = "/kaggle/working/rclone.conf"
with open(conf_path, "w") as f:
    f.write(base64.b64decode(UserSecretsClient().get_secret("RCLONE_CONF")).decode())
os.environ["RCLONE_CONFIG"] = conf_path

remotes = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True).stdout.strip().split("\n")
remote  = remotes[0] if remotes and remotes[0] else ""
# Guard: an empty remote makes REMOTE_NAME a *local* relative path, so rclone
# would silently copy checkpoints into /kaggle/working instead of Drive.
assert remote.endswith(":"), (
    f"No rclone remote found (rclone listremotes -> {remotes!r}). "
    "Check the RCLONE_CONF secret. Without a remote, checkpoints would be written "
    "LOCALLY to /kaggle/working, not Drive."
)
REMOTE_NAME = remote + f"{NOTEBOOK_NAME}/ckpts/"

out = subprocess.run(["rclone", "mkdir", REMOTE_NAME], capture_output=True, text=True)
print("remote drive ──>", REMOTE_NAME, "(ok)" if out.returncode == 0 else f"ERROR: {out.stderr.strip()}")


In [ ]:
# ── Verbosity ─────────────────────────────────────────────────────────────────
#
#   "epoch"      — one summary line per epoch (train/val/test loss + R²)
#   "batch"      — also prints running-average loss every 20 batches
#   "diagnostic" — per-batch NaN/Inf check with full tensor stats on the first
#                  10 batches; use when debugging numerical issues
#
VERBOSITY = "batch"  

import sys, os
sys.path.insert(0, f"/kaggle/working/{NOTEBOOK_NAME}")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # prevents fragmentation
os.environ["PYTHONPATH"] = f"/kaggle/working/{NOTEBOOK_NAME}" # inherited by mp.spawn workers

# clear python cache
for mod in list(sys.modules.keys()):
    if "ScatterNet" in mod or "train" in mod:
        del sys.modules[mod]

from ScatterNet.config import RunConfig, DEFAULT_BUCKETS
from train import main

# bin 58 contains molecules up to 78,819 atoms; even with 2-GPU TP the shard
# (~39k atoms) exhausts T4 memory. Drop it so max molecule becomes 6,046 atoms.
SAFE_BUCKETS = [b for b in DEFAULT_BUCKETS if b[1] <= 6046]

In [ ]:
cfg = RunConfig(

    # --- paths ---
    hdf5           = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5",       # HDF5 dataset
    db             = f"/kaggle/working/{NOTEBOOK_NAME}/Preprocess/scatternet", # SQLite encoding stem
    ckpt_best      = "/kaggle/working/scatternet_best.pt",                     # LOCAL staging: best-val model (auto-copied to ckpt_rclone_dest)
    ckpt_resume    = "/kaggle/working/scatternet_resume.pt",                   # LOCAL staging: latest state for crash-resume (copied to Drive every ckpt_interval_sec + each epoch)
    metrics        = "/kaggle/working/scatternet_metrics.json",                # LOCAL staging: per-epoch loss/R2 log (auto-copied to ckpt_rclone_dest)
    resume         = None,                                                     # path to resume from, or None

    # --- checkpointing / crash-safety ---
    ckpt_rclone_dest  = f"{REMOTE_NAME}", # rclone dest; checkpoints copied here after every save
    ckpt_interval_sec = 600,              # save a mid-epoch resume point every ckpt_interval_sec seconds

    # --- model ---
    lambda_1       = 128,  # atom embedding dimension
    lambda_2       = 5,    # message passing rounds
    lambda_3       = 128,  # OutputHead hidden width
    lambda_4       = 4,    # MLP halving steps (2^lambda_4 <= lambda_3)
    lambda_5       = 128,  # Random Fourier Features
    msg_seed       = 42,   # RFF frequency matrix seed
    atm_chunk      = 64,   # atoms per M-chunk
    mol_chunk      = 64,   # molecules per N-chunk
    eps_embd       = 1e-8, # numerical floor in Embed
    eps_msgp       = 1e-3, # floor in MessagePass (Å)

    # --- loss ---
    lambda_6       = 0.1,   # form-factor penalty weight
    lambda_7       = 0.1,   # sigma inverse-L1 regularisation weight

    # --- training ---
    lr             = 3e-4,  # Adam learning rate
    weight_decay   = 1e-5,  # Adam L2 weight decay
    grad_clip      = 1.0,   # max gradient norm
    epochs         = 50,    # epochs to train
    batcher_seed   = 0,     # train/val/test split seed
    atom_size_ceil = 78819, # max atoms per batch — MUST be large; small values create too many batches
    num_workers    = 3,     # DataLoader workers
    max_batches    = None,  # cap batches per epoch (None = no limit)
    verbosity      = VERBOSITY,

    # --- data ---
    buckets        = SAFE_BUCKETS,
)

main(cfg)

In [ ]:
# ── PROFILER / diagnostic run ───────────────────────────────────────────────
# Uncomment to profile instead of train. Stops after the profiling window
# (no eval/checkpoint). Every rank is wrapped in torch.profiler (per-rank
# TensorBoard trace) AND a lightweight section timer that prints, per rank, a
# data-wait / H2D / forward / backward / grad-allreduce / clip / step breakdown
# plus the heaviest batches — compare ranks to spot tensor-parallel skew.

# cfg = RunConfig(

#     # --- paths ---
#     hdf5              = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5",
#     db                = f"/kaggle/working/{NOTEBOOK_NAME}/Preprocess/scatternet",
#     ckpt_best         = "/kaggle/working/scatternet_best.pt",
#     ckpt_resume       = "/kaggle/working/scatternet_resume.pt",
#     metrics           = "/kaggle/working/scatternet_metrics.json",
#     resume            = None,

#     # --- checkpointing / crash-safety ---
#     ckpt_rclone_dest  = f"{REMOTE_NAME}",
#     ckpt_interval_sec = 600,

#     # --- model ---
#     lambda_1          = 128,
#     lambda_2          = 5,
#     lambda_3          = 128,
#     lambda_4          = 4,
#     lambda_5          = 128,
#     msg_seed          = 42,
#     atm_chunk         = 64,
#     mol_chunk         = 64,
#     eps_embd          = 1e-8,
#     eps_msgp          = 1e-3,

#     # --- loss ---
#     lambda_6          = 0.1,
#     lambda_7          = 0.1,

#     # --- training ---
#     lr                = 3e-4,
#     weight_decay      = 1e-5,
#     grad_clip         = 1.0,
#     epochs            = 50,
#     batcher_seed      = 0,
#     atom_size_ceil    = 78819,
#     num_workers       = 3,
#     max_batches       = None,
#     verbosity         = VERBOSITY,

#     # --- data ---
#     buckets           = SAFE_BUCKETS,

#     # --- profiler (diagnostic run) ---
#     profiler          = True,  # per-rank torch.profiler + section timers
#     prof_warmup       = 2,     # warmup batches (profiled, discarded)
#     prof_active       = 20,    # recorded batches; bump for representative stats
# )

# main(cfg)


In [ ]:
# # ── Viewing the profiler ────────────────────────────────────────────────────
# # You usually DON'T need this cell: the diagnostic run (main(cfg) above) already
# # prints the per-rank section breakdown AND a top-GPU-ops table to stdout, which
# # is all the speed info you need. TensorBoard's inline view often hangs through
# # Kaggle's proxy, so prefer the printed output.
# #
# # If you want the interactive timeline, download the trace and open it in
# # https://ui.perfetto.dev (drag-and-drop) or chrome://tracing — this avoids the
# # Kaggle iframe entirely:
# import glob, os
# traces = glob.glob(os.path.abspath("./profiler_trace") + "/**/*.pt.trace.json*", recursive=True)
# print(f"{len(traces)} trace file(s) — download from the Output tab, open in ui.perfetto.dev:")
# for t in traces:
#     print("  ", t)

# # Optional (often hangs on Kaggle): kill any stale server first, then relaunch.
# # !pkill -f tensorboard
# # %reload_ext tensorboard
# # %tensorboard --logdir {os.path.abspath("./profiler_trace")}


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RESUME after a crash / session timeout
# ═══════════════════════════════════════════════════════════════════════════
# Training saves a resume checkpoint to Drive every ckpt_interval_sec. To resume:
#   1. Re-run the cells above EXCEPT the "Fresh training run" cell:
#        deps -> git-lfs -> git pull -> rclone setup -> the RunConfig cell
#        (the RunConfig cell defines `cfg` and imports `main`; it no longer
#         starts training, so running it is safe).
#   2. Run THIS cell. It pulls the latest checkpoint from Drive and continues
#      from the saved batch. Mid-epoch resume is exact (the per-epoch seed makes
#      the shuffle reproducible), so you lose at most ckpt_interval_sec of work.
import dataclasses, subprocess

# pull the last checkpoint back from Drive
subprocess.run(["rclone", "copy", f"{REMOTE_NAME}/scatternet_resume.pt", "/kaggle/working/"], check=True)

resume_cfg = dataclasses.replace(cfg, resume="/kaggle/working/scatternet_resume.pt")
main(resume_cfg)
